# Team 04 — Intent Extraction (Phase 0)

Visual proof that the agent **comprehends short prompts** into a typed `DesignBrief`
instead of relying on long hand-written prompt rules (BACKEND_PLAN.md, Phase 0).

Sections:
1. A varied prompt set (terse / verbose / vague / contradictory)
2. Deterministic **fallback** extraction (regex, no LLM) — always runs
3. No-invention check — vague prompts yield `auto`/`null`, not made-up numbers
4. **LLM** extraction (optional — runs only if Team 04 LLM env is configured)
5. Canonical **SiteModel** (sides, corners, buildable zone)


In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (
    workspace_root,
    workspace_root.parent,
    workspace_root / 'team_04',
    workspace_root.parent / 'team_04',
)
TEAM_ROOT = next((p for p in candidate_roots if (p / 'agent').exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError('Run from workspace root, team_04, or team_04/test_notebooks.')
if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))

from agent.brief import extract_brief_fallback, resolve_brief
from agent.tools.site_model import build_site_model
print('Team root:', TEAM_ROOT)


## 1. Prompt set

A spread of phrasings the agent should comprehend without bespoke rules.

In [ ]:
PROMPTS = [
    # (label, prompt, layout_payload)
    ('terse',        'Two U-shaped apartment buildings with a quiet courtyard and parking.', {}),
    ('verbose',      'I would like to develop the site with a single L-shaped office building '
                     'of roughly 3000 square meters, six storeys tall, oriented so the best '
                     'daylight reaches the workspaces. Daylight matters most to me.', {}),
    ('digits',       'Place 3 residential buildings and rotate the building by 45 degrees.', {}),
    ('vague',        'Make something nice with a bit of greenery on the plot.', {}),
    ('contradictory','One building that is both U-shaped and a bar building.', {}),
    ('layout-count', 'Design an office building facing the street.',
                     {'target_building_count': 2, 'building_intents': ['calm west edge', 'face the road']}),
    ('courtyard',    'A single O-shaped ring building around a sunny private courtyard.', {}),
    ('bar+view',     'A bar building (I-shape) that maximises the view to the park.', {}),
]
len(PROMPTS)


## 2. Fallback extraction (regex, no LLM)

Always available — keeps unit tests and offline runs working.

In [ ]:
def brief_row(label, brief):
    b0 = brief.buildings[0] if brief.buildings else None
    return {
        'label': label,
        'count': brief.building_count,
        'shape_0': b0.shape_preference if b0 else None,
        'area_0': b0.footprint_area_sqm if b0 else None,
        'storeys_0': b0.storeys if b0 else None,
        'use_0': b0.use if b0 else None,
        'courtyard': brief.courtyard_requested,
        'parking': brief.parking_requested,
        'rotation': brief.requested_rotation_deg,
        'view_w': brief.view_weight,
        'sun_w': brief.sun_weight,
        'align_w': brief.alignment_weight,
        'source': brief.source,
    }

rows = [brief_row(label, extract_brief_fallback(prompt, layout)) for label, prompt, layout in PROMPTS]

try:
    import pandas as pd
    df = pd.DataFrame(rows).set_index('label')
    display(df)
except ImportError:
    for r in rows:
        print(r)


## 3. No-invention check

The vague prompt must **not** fabricate a shape or area — it stays `auto` / `None`.
(The richer LLM extractor additionally records *why* in `ambiguities`; the regex
fallback simply abstains.)

In [ ]:
vague = extract_brief_fallback('Make something nice with a bit of greenery on the plot.', {})
print('count    :', vague.building_count)
print('shape    :', vague.buildings[0].shape_preference, '(expected: auto)')
print('area     :', vague.buildings[0].footprint_area_sqm, '(expected: None)')
print('rotation :', vague.requested_rotation_deg, '(expected: None)')
assert vague.buildings[0].shape_preference == 'auto'
assert vague.buildings[0].footprint_area_sqm is None
print()
print('OK - nothing invented for a vague prompt.')


## 4. LLM extraction (optional)

Runs only if Team 04 LLM environment variables are configured. This is where the
'less prompt, more comprehension' payoff shows: the model resolves count from natural
phrasing, distinguishes per-building intent, raises the weight of whatever the user
emphasises, and lists genuine `ambiguities` (e.g. the contradictory prompt) instead of guessing.

In [ ]:
llm_engine = None
try:
    from langchain_openai import ChatOpenAI
    from agent.config import load_settings
    from agent.decision_engine import OpenAIDecisionEngine
    settings = load_settings()
    llm = ChatOpenAI(api_key=settings.api_key, base_url=settings.base_url,
                     model=settings.llm_model, timeout=settings.request_timeout_seconds, temperature=0)
    llm_engine = OpenAIDecisionEngine(
        llm=llm,
        decision_provider=settings.decision_llm_provider, decision_model=settings.decision_llm_model,
        report_provider=settings.report_llm_provider, report_model=settings.report_llm_model,
    )
    print('LLM engine ready:', settings.llm_model)
except Exception as exc:
    print('Skipping live LLM extraction (env not configured):', type(exc).__name__, exc)


In [ ]:
if llm_engine is not None:
    llm_rows = []
    for label, prompt, layout in PROMPTS:
        brief = resolve_brief(llm_engine, prompt, layout)  # LLM; falls back to regex on failure
        row = brief_row(label, brief)
        row['ambiguities'] = ' | '.join(brief.ambiguities)
        llm_rows.append(row)
    try:
        import pandas as pd
        display(pd.DataFrame(llm_rows).set_index('label'))
    except ImportError:
        for r in llm_rows:
            print(r)
else:
    print('No LLM engine - section 2 (fallback) is the offline baseline.')


## 5. Canonical SiteModel

The structure every later phase (sun, roads, grid, parking) reads from.

In [ ]:
SITE = [[0, 0, 0], [120, 0, 0], [120, 80, 0], [0, 80, 0], [0, 0, 0]]
model = build_site_model(SITE, {'edge_road_widths': {0: 20.0}})  # side 0 fronts a 20 m road
print('available     :', model['available'])
print('corners       :', len(model['corners']))
print('sides         :', len(model['sides']))
print('site area     :', model['setbacks']['site_area_sqm'])
print('buildable area:', model['setbacks']['buildable_area_sqm'])
print('phase slots   : roads=%s grid=%s sun=%s' % (model['roads'], model['grid'], model['sun']))


In [ ]:
try:
    import matplotlib.pyplot as plt
    sx = [p[0] for p in SITE]; sy = [p[1] for p in SITE]
    bz = model['setbacks']['buildable_boundary']
    bx = [p[0] for p in bz]; by = [p[1] for p in bz]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(sx, sy, 'k-', lw=2, label='site boundary')
    ax.plot(bx, by, 'g--', lw=1.5, label='buildable zone (after setbacks)')
    for corner in model['corners']:
        px, py = corner['point'][0], corner['point'][1]
        ax.plot(px, py, 'ko')
        ax.annotate(corner['label'], (px, py), textcoords='offset points', xytext=(5, 5))
    for side in model['sides']:
        mp = side.get('midpoint') or side.get('mid_point')
        if mp:
            ax.annotate(side['label'], (mp[0], mp[1]), color='blue', ha='center')
    ax.set_aspect('equal'); ax.legend(); ax.set_title('SiteModel: sides, corners, buildable zone')
    plt.show()
except ImportError:
    print('matplotlib not installed in this kernel; numbers above still confirm the model.')
